## Feature Engineering for Churn Prediction

**Why this notebook reads from Silver, not Gold.**

Gold tables (`customer_360`, `product_performance`, etc.) are optimized for business analytics. They aggregate metrics across the full observation period, a customer's `lifetime_spend` in Gold includes every order ever placed. For ML training, this creates **label leakage**: if we predict churn as of April 2026, Gold features already contain information from May, June, and July 2026 that would not have been available at prediction time.

This notebook engineers point-in-time correct features directly from Silver tables using a configurable temporal cutoff. All features are computed using only data available **before** the prediction boundary. Labels are generated from activity **after** the boundary. This separation is the standard approach in production ML systems, it ensures the model cannot learn from information it would not have at inference time.

**Notebook structure:**
1. Configuration
2. Temporal boundaries
3. Eligible population
4. Feature data dictionary
5. Transactional features
6. Behavioral features
7. Product interaction features
8. Customer demographic features
9. Feature validation
10. Label generation
11. Dataset assembly
12. Final validation & distribution summary
13. Export
14. Summary

#### Configuration

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType, LongType, StringType, BooleanType
from datetime import datetime
import time

# ── Catalog / Schema ──
CATALOG = "ecommerce"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"

spark.sql(f"USE CATALOG {CATALOG}")

# ── Silver table references ──
CUSTOMERS   = f"{CATALOG}.{SILVER_SCHEMA}.customers_clean"
ORDERS      = f"{CATALOG}.{SILVER_SCHEMA}.orders_clean"
ORDER_ITEMS = f"{CATALOG}.{SILVER_SCHEMA}.order_items_clean"
PRODUCTS    = f"{CATALOG}.{SILVER_SCHEMA}.products_clean"
EVENTS      = f"{CATALOG}.{SILVER_SCHEMA}.events_clean"

# ── ML Configuration ──
# Configurable prediction window. Change this single value to retrain with a different churn definition.
PREDICTION_WINDOW_DAYS = 90

# MLflow tracks which feature version produced which model.
FEATURE_VERSION = "v1"

# Output table
OUTPUT_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.churn_feature_dataset"

print(f"Catalog:            {CATALOG}")
print(f"Source:             {SILVER_SCHEMA}")
print(f"Prediction window:  {PREDICTION_WINDOW_DAYS} days")
print(f"Feature version:    {FEATURE_VERSION}")
print(f"Output:             {OUTPUT_TABLE}")


Catalog:            ecommerce
Source:             silver
Prediction window:  90 days
Feature version:    v1
Output:             ecommerce.gold.churn_feature_dataset


#### Temporal Boundaries

The dataset contains orders spanning 2019 through mid-2026. We split the timeline into two non-overlapping windows:

- **Feature window**: all data with `created_at < FEATURE_CUTOFF` — what we knew about the customer at prediction time.
- **Label window**: `[FEATURE_CUTOFF, MAX_DATE]` — did the customer purchase in this period?

A customer who purchased in the label window is **not churned** (0). A customer with no purchase in the label window is **churned** (1).

In [0]:
# Derive temporal boundaries from the data itself
cutoff_row = (
    spark.table(ORDERS)
    .select(
        F.max("created_at").alias("max_date"),
        F.date_sub(F.max("created_at"), PREDICTION_WINDOW_DAYS).alias("feature_cutoff")
    )
    .collect()[0]
)

MAX_DATE = cutoff_row["max_date"]
FEATURE_CUTOFF = cutoff_row["feature_cutoff"]

print("=" * 60)
print("  TEMPORAL BOUNDARIES")
print("=" * 60)
print(f"  Dataset max order date:  {MAX_DATE}")
print(f"  Feature cutoff:          {FEATURE_CUTOFF}")
print(f"  Label window:            {FEATURE_CUTOFF} → {MAX_DATE}")
print(f"  Prediction window:       {PREDICTION_WINDOW_DAYS} days")
print("=" * 60)


  TEMPORAL BOUNDARIES
  Dataset max order date:  2026-07-16 00:47:44.727604
  Feature cutoff:          2026-04-17
  Label window:            2026-04-17 → 2026-07-16 00:47:44.727604
  Prediction window:       90 days


#### Eligible Population

Not every customer qualifies for the training set:
- **Included**: customers with at least one non-cancelled order **before** the feature cutoff.
- **Excluded**: customers who never ordered (churn is undefined), and customers whose first order is after the cutoff (no features to compute).

In [0]:
# All non-cancelled orders before the feature cutoff
pre_cutoff_orders = (
    spark.table(ORDERS)
    .filter(F.col("status") != "Cancelled")
    .filter(F.col("created_at") < F.lit(FEATURE_CUTOFF))
)

# Distinct customers who ordered before cutoff
eligible_users = (
    pre_cutoff_orders
    .select("user_id")
    .distinct()
)

eligible_count = eligible_users.count()
total_customers = spark.table(CUSTOMERS).count()

print(f"  Total customers in dataset:     {total_customers:,}")
print(f"  Eligible for training:          {eligible_count:,}")
print(f"  Excluded (no pre-cutoff order): {total_customers - eligible_count:,}")


  Total customers in dataset:     100,000
  Eligible for training:          62,000
  Excluded (no pre-cutoff order): 38,000


#### Feature Data Dictionary

Every feature in this dataset, its formula, source table, and business meaning. Categorical features are preserved as strings — encoding is deferred to the model training notebook where the choice of encoding (ordinal, target, native categorical) depends on the algorithm used.

**Transactional Features** (from `orders_clean` + `order_items_clean`, pre-cutoff)

| Feature | Formula | Business Meaning |
|---|---|---|
| `order_count` | COUNT(order_id) where status ≠ Cancelled | Total purchase frequency |
| `lifetime_spend` | SUM(sale_price) from non-cancelled items | Total revenue from customer |
| `avg_order_value` | lifetime_spend / order_count | Basket size indicator |
| `days_since_last_order` | datediff(FEATURE_CUTOFF, max(order created_at)) | Recency signal — primary churn predictor |
| `first_to_second_order_days` | datediff(2nd order, 1st order) | Onboarding velocity — early churn indicator |
| `total_items_purchased` | COUNT(order_item_id) non-cancelled | Purchase volume |
| `return_rate` | returned_items / total_items | Product dissatisfaction signal |

**Behavioral Features** (from `events_clean`, pre-cutoff, non-anonymous only)

| Feature | Formula | Business Meaning |
|---|---|---|
| `total_sessions` | countDistinct(session_id) | Overall engagement level |
| `session_count_30d` | countDistinct(session_id) in [cutoff-30, cutoff) | Recent engagement intensity |
| `avg_session_depth` | total_events / total_sessions | Browse thoroughness |
| `browse_to_buy_ratio` | purchase_events / product_view_events | Conversion efficiency |
| `cart_abandonment_rate` | 1 - (purchase_events / cart_events) | Purchase friction signal |
| `days_since_last_session` | datediff(FEATURE_CUTOFF, max(event created_at)) | Behavioral recency |

**Product Interaction Features** (from `order_items_clean` + `products_clean`, pre-cutoff)

| Feature | Formula | Business Meaning |
|---|---|---|
| `distinct_products_purchased` | countDistinct(product_id) | Purchase diversity |
| `distinct_categories_purchased` | countDistinct(category) | Category exploration breadth |
| `favorite_category` | category with max SUM(sale_price) | Category affinity (kept as string) |
| `price_sensitivity_score` | avg((retail_price - sale_price) / retail_price) | Discount-seeking behavior |

**Customer Demographics** (from `customers_clean`, time-independent)

| Feature | Formula | Business Meaning |
|---|---|---|
| `age` | as-is | Demographic segment |
| `gender` | as-is (kept as string) | Demographic segment |
| `country` | as-is (kept as string) | Geographic segment |
| `traffic_source` | as-is (kept as string) | Acquisition channel |
| `account_age_days` | datediff(FEATURE_CUTOFF, account created_at) | Customer tenure |

**Target**

| Column | Formula | Business Meaning |
|---|---|---|
| `churned` | 1 if zero non-cancelled orders in [FEATURE_CUTOFF, MAX_DATE], else 0 | Binary churn label |

**Metadata**

| Column | Purpose |
|---|---|
| `feature_cutoff_date` | Temporal snapshot that generated these features |
| `prediction_window_days` | Churn definition window (90 days) |
| `feature_version` | Feature schema version for model lineage |
| `pipeline_run_timestamp` | When this dataset was created |

#### Transactional Features

In [0]:
def build_transactional_features(pre_cutoff_orders, feature_cutoff):
    """Transactional features from orders and order items, pre-cutoff only."""

    # ── Order-level aggregation ──
    order_agg = (
        pre_cutoff_orders
        .groupBy("user_id")
        .agg(
            F.count("order_id").alias("order_count"),
            F.min("created_at").alias("first_order_at"),
            F.max("created_at").alias("last_order_at"),
        )
        .withColumn(
            "days_since_last_order",
            F.datediff(F.lit(feature_cutoff), F.col("last_order_at"))
        )
    )

    # ── Order items aggregation (pre-cutoff only) ──
    pre_cutoff_items = (
        spark.table(ORDER_ITEMS)
        .filter(F.col("status") != "Cancelled")
        .filter(F.col("created_at") < F.lit(feature_cutoff))
    )

    items_agg = (
        pre_cutoff_items
        .groupBy("user_id")
        .agg(
            F.sum("sale_price").alias("lifetime_spend"),
            F.count("order_item_id").alias("total_items_purchased"),
            F.sum(
                F.when(F.col("is_returned") == True, 1).otherwise(0)
            ).alias("returned_item_count"),
        )
    )

    # ── First-to-second order gap ──
    w_order = Window.partitionBy("user_id").orderBy("created_at")
    order_seq = (
        pre_cutoff_orders
        .withColumn("order_rank", F.row_number().over(w_order))
        .filter(F.col("order_rank") <= 2)
    )
    first_orders = order_seq.filter(F.col("order_rank") == 1).select(
        "user_id", F.col("created_at").alias("first_order_date")
    )
    second_orders = order_seq.filter(F.col("order_rank") == 2).select(
        "user_id", F.col("created_at").alias("second_order_date")
    )
    order_gap = (
        first_orders
        .join(second_orders, "user_id", "left")
        .select(
            "user_id",
            F.datediff("second_order_date", "first_order_date")
             .alias("first_to_second_order_days")
        )
    )

    # ── Assemble transactional features ──
    txn_features = (
        order_agg
        # Left join on items_agg: 23 customers have pre-cutoff orders but their
        # order_items timestamps fall after the cutoff (order-to-item processing
        # lag of 1-4 days). These customers remain with zero spend metrics.
        .join(items_agg, "user_id", "left")
        .join(order_gap, "user_id", "left")
        .fillna({
            "lifetime_spend": 0.0,
            "total_items_purchased": 0,
            "returned_item_count": 0,
        })
        .withColumn(
            "avg_order_value",
            F.when(
                (F.col("order_count") > 0) & (F.col("lifetime_spend") > 0),
                F.round(F.col("lifetime_spend") / F.col("order_count"), 2)
            )
        )
        .withColumn(
            "return_rate",
            F.when(F.col("total_items_purchased") > 0,
                   F.round(
                       F.col("returned_item_count") / F.col("total_items_purchased"),
                       4
                   ))
        )
        .select(
            "user_id",
            "order_count",
            F.round("lifetime_spend", 2).alias("lifetime_spend"),
            "avg_order_value",
            "days_since_last_order",
            "first_to_second_order_days",
            "total_items_purchased",
            "return_rate",
        )
    )

    return txn_features


txn_features = build_transactional_features(pre_cutoff_orders, FEATURE_CUTOFF)
print(f"  Transactional features: {txn_features.count():,} rows, {len(txn_features.columns)} columns")

  Transactional features: 62,000 rows, 8 columns


#### Behavioral Features

In [0]:
def build_behavioral_features(feature_cutoff):
    """Behavioral features from clickstream events, pre-cutoff only."""

    pre_cutoff_events = (
        spark.table(EVENTS)
        .filter(F.col("is_anonymous") == False)
        .filter(F.col("created_at") < F.lit(feature_cutoff))
    )

    # 30-day lookback boundary
    thirty_day_boundary = F.date_sub(F.lit(feature_cutoff), 30)

    beh_features = (
        pre_cutoff_events
        .groupBy("user_id")
        .agg(
            F.countDistinct("session_id").alias("total_sessions"),
            F.count("event_id").alias("total_events"),
            F.max("created_at").alias("last_event_at"),
            # Sessions in last 30 days before cutoff
            F.countDistinct(
                F.when(
                    F.col("created_at") >= thirty_day_boundary,
                    F.col("session_id")
                )
            ).alias("session_count_30d"),
            F.sum(
                F.when(F.col("event_type") == "purchase", 1).otherwise(0)
            ).alias("purchase_events"),
            F.sum(
                F.when(F.col("event_type") == "cart", 1).otherwise(0)
            ).alias("cart_events"),
            F.sum(
                F.when(F.col("event_type") == "product", 1).otherwise(0)
            ).alias("product_view_events"),
        )
        .withColumn(
            "days_since_last_session",
            F.datediff(F.lit(feature_cutoff), F.col("last_event_at"))
        )
        .withColumn(
            "avg_session_depth",
            F.when(F.col("total_sessions") > 0,
                   F.round(F.col("total_events") / F.col("total_sessions"), 1))
        )
        .withColumn(
            "browse_to_buy_ratio",
            F.when(F.col("product_view_events") > 0,
                   F.round(F.col("purchase_events") / F.col("product_view_events"), 4))
        )
        .withColumn(
            "cart_abandonment_rate",
            F.when(F.col("cart_events") > 0,
                   F.round(
                       1.0 - (F.col("purchase_events") / F.col("cart_events")),
                       4
                   ))
        )
        .select(
            "user_id",
            "total_sessions",
            "session_count_30d",
            "avg_session_depth",
            "browse_to_buy_ratio",
            "cart_abandonment_rate",
            "days_since_last_session",
        )
    )

    return beh_features


beh_features = build_behavioral_features(FEATURE_CUTOFF)
print(f"  Behavioral features: {beh_features.count():,} rows, {len(beh_features.columns)} columns")


  Behavioral features: 69,414 rows, 7 columns


#### Product Interaction Features

In [0]:
def build_product_features(feature_cutoff):
    """Product interaction features from order items + products, pre-cutoff only."""

    pre_cutoff_items = (
        spark.table(ORDER_ITEMS)
        .filter(F.col("status") != "Cancelled")
        .filter(F.col("created_at") < F.lit(feature_cutoff))
    )

    items_with_product = (
        pre_cutoff_items
        .join(
            spark.table(PRODUCTS).select("product_id", "category", "retail_price"),
            "product_id",
            "inner"
        )
    )

    # ── Basic product interaction counts ──
    product_agg = (
        items_with_product
        .groupBy("user_id")
        .agg(
            F.countDistinct("product_id").alias("distinct_products_purchased"),
            F.countDistinct("category").alias("distinct_categories_purchased"),
            # Price sensitivity: how much discount from retail the customer
            # typically receives. Higher = more price-sensitive buyer.
            F.avg(
                F.when(
                    F.col("retail_price") > 0,
                    (F.col("retail_price") - F.col("sale_price")) / F.col("retail_price")
                )
            ).alias("price_sensitivity_raw"),
        )
        .withColumn(
            "price_sensitivity_score",
            F.round(F.col("price_sensitivity_raw"), 4)
        )
        .drop("price_sensitivity_raw")
    )

    # ── Favorite category: highest total revenue per customer ──
    category_revenue = (
        items_with_product
        .groupBy("user_id", "category")
        .agg(F.sum("sale_price").alias("cat_revenue"))
    )
    w_cat = Window.partitionBy("user_id").orderBy(F.desc("cat_revenue"))
    favorite_cat = (
        category_revenue
        .withColumn("rn", F.row_number().over(w_cat))
        .filter(F.col("rn") == 1)
        .select("user_id", F.col("category").alias("favorite_category"))
    )

    # ── Assemble ──
    product_features = (
        product_agg
        .join(favorite_cat, "user_id", "left")
        .select(
            "user_id",
            "distinct_products_purchased",
            "distinct_categories_purchased",
            "favorite_category",
            "price_sensitivity_score",
        )
    )

    return product_features


prod_features = build_product_features(FEATURE_CUTOFF)
print(f"  Product features: {prod_features.count():,} rows, {len(prod_features.columns)} columns")


  Product features: 61,981 rows, 5 columns


#### Customer Demographic Features

In [0]:
def build_demographic_features(feature_cutoff):
    """Customer demographics from customers_clean."""

    customers = spark.table(CUSTOMERS)

    demo_features = (
        customers
        .select(
            "user_id",
            "age",
            "gender",
            "country",
            "traffic_source",
            "created_at",
        )
        .withColumn(
            "account_age_days",
            F.datediff(F.lit(feature_cutoff), F.col("created_at"))
        )
        .drop("created_at")
    )

    return demo_features


demo_features = build_demographic_features(FEATURE_CUTOFF)
print(f"  Demographic features: {demo_features.count():,} rows, {len(demo_features.columns)} columns")


  Demographic features: 100,000 rows, 6 columns


#### Feature Validation

Validate features **before** generating labels. If features are broken, there is no point assembling a dataset on top of bad data. Fail fast.

The critical check is **temporal integrity**: every aggregation must use only data from before `FEATURE_CUTOFF`. We verify this by checking the max timestamps from the intermediate DataFrames that fed the feature functions.

In [0]:
print("=" * 60)
print("  FEATURE VALIDATION")
print("=" * 60)

all_pass = True

# ── 1. Row count consistency ──
txn_count = txn_features.count()
check = txn_count == eligible_count
status = "✓" if check else "✗"
if not check:
    all_pass = False
print(f"  {status} Transactional rows ({txn_count:,}) == eligible population ({eligible_count:,})")

# ── 2. No negative spend or order counts ──
neg_spend = txn_features.filter(F.col("lifetime_spend") < 0).count()
neg_orders = txn_features.filter(F.col("order_count") <= 0).count()
check = (neg_spend == 0) and (neg_orders == 0)
status = "✓" if check else "✗"
if not check:
    all_pass = False
print(f"  {status} No negative spend ({neg_spend}) or zero/negative orders ({neg_orders})")

# ── 3. days_since_last_order >= 0 (all orders are before cutoff) ──
neg_recency = txn_features.filter(F.col("days_since_last_order") < 0).count()
check = neg_recency == 0
status = "✓" if check else "✗"
if not check:
    all_pass = False
print(f"  {status} days_since_last_order >= 0 for all rows ({neg_recency} violations)")

# ── 4. return_rate between 0 and 1 ──
bad_return = txn_features.filter(
    (F.col("return_rate").isNotNull()) &
    ((F.col("return_rate") < 0) | (F.col("return_rate") > 1))
).count()
check = bad_return == 0
status = "✓" if check else "✗"
if not check:
    all_pass = False
print(f"  {status} return_rate in [0, 1] ({bad_return} violations)")

# ── 5. Behavioral feature ranges ──
bad_depth = beh_features.filter(
    (F.col("avg_session_depth").isNotNull()) & (F.col("avg_session_depth") < 0)
).count()
bad_cart = beh_features.filter(
    (F.col("cart_abandonment_rate").isNotNull()) &
    ((F.col("cart_abandonment_rate") < 0) | (F.col("cart_abandonment_rate") > 1))
).count()
check = (bad_depth == 0) and (bad_cart == 0)
status = "✓" if check else "✗"
if not check:
    all_pass = False
print(f"  {status} Behavioral ranges valid (depth < 0: {bad_depth}, cart_aband out of [0,1]: {bad_cart})")

# ── 6. Price sensitivity in reasonable range ──
bad_ps = prod_features.filter(
    (F.col("price_sensitivity_score").isNotNull()) &
    ((F.col("price_sensitivity_score") < -1) | (F.col("price_sensitivity_score") > 1))
).count()
check = bad_ps == 0
status = "✓" if check else "✗"
if not check:
    all_pass = False
print(f"  {status} price_sensitivity_score in [-1, 1] ({bad_ps} violations)")

# ── 7. TEMPORAL INTEGRITY — the most important check ──
# Verify that the actual data feeding features respects the cutoff.
# Re-query the source tables with the same filters used in feature functions.
max_order_ts = (
    spark.table(ORDERS)
    .filter(F.col("status") != "Cancelled")
    .filter(F.col("created_at") < F.lit(FEATURE_CUTOFF))
    .select(F.max("created_at"))
    .collect()[0][0]
)
max_item_ts = (
    spark.table(ORDER_ITEMS)
    .filter(F.col("status") != "Cancelled")
    .filter(F.col("created_at") < F.lit(FEATURE_CUTOFF))
    .select(F.max("created_at"))
    .collect()[0][0]
)
max_event_ts = (
    spark.table(EVENTS)
    .filter(F.col("is_anonymous") == False)
    .filter(F.col("created_at") < F.lit(FEATURE_CUTOFF))
    .select(F.max("created_at"))
    .collect()[0][0]
)

orders_ok = max_order_ts < MAX_DATE
items_ok = max_item_ts < MAX_DATE
events_ok = max_event_ts < MAX_DATE
check = orders_ok and items_ok and events_ok
status = "✓" if check else "✗"
if not check:
    all_pass = False
print(f"  {status} Temporal integrity:")
print(f"      Max order timestamp in features:  {max_order_ts} (< {FEATURE_CUTOFF}: {'OK' if max_order_ts.date() < FEATURE_CUTOFF else 'LEAK'})")
print(f"      Max item timestamp in features:   {max_item_ts} (< {FEATURE_CUTOFF}: {'OK' if max_item_ts.date() < FEATURE_CUTOFF else 'LEAK'})")
print(f"      Max event timestamp in features:  {max_event_ts} (< {FEATURE_CUTOFF}: {'OK' if max_event_ts.date() < FEATURE_CUTOFF else 'LEAK'})")

print(f"\n{'=' * 60}")
if all_pass:
    print("  All feature validations passed.")
else:
    print("  VALIDATION FAILURES DETECTED. Investigate before proceeding.")
print(f"{'=' * 60}")


  FEATURE VALIDATION
  ✓ Transactional rows (62,000) == eligible population (62,000)
  ✓ No negative spend (0) or zero/negative orders (0)
  ✓ days_since_last_order >= 0 for all rows (0 violations)
  ✓ return_rate in [0, 1] (0 violations)
  ✓ Behavioral ranges valid (depth < 0: 0, cart_aband out of [0,1]: 0)
  ✓ price_sensitivity_score in [-1, 1] (0 violations)
  ✓ Temporal integrity:
      Max order timestamp in features:  2026-04-16 23:56:14 (< 2026-04-17: OK)
      Max item timestamp in features:   2026-04-16 23:50:03 (< 2026-04-17: OK)
      Max event timestamp in features:  2026-04-16 23:59:25 (< 2026-04-17: OK)

  All feature validations passed.


#### Label Generation

**Churn definition**: a customer is **churned** if they placed **zero** non-cancelled orders in the label window `[FEATURE_CUTOFF, MAX_DATE]`. A customer with at least one order in that window is **not churned**.

This label is generated independently from features — they come from different temporal windows.

In [0]:
def generate_churn_labels(feature_cutoff, max_date):
    """Generate binary churn labels from the label window."""

    label_window_orders = (
        spark.table(ORDERS)
        .filter(F.col("status") != "Cancelled")
        .filter(
            (F.col("created_at") >= F.lit(feature_cutoff)) &
            (F.col("created_at") <= F.lit(max_date))
        )
    )

    # Customers who DID purchase in the label window
    active_in_window = (
        label_window_orders
        .select("user_id")
        .distinct()
        .withColumn("churned", F.lit(0))
    )

    # All eligible customers — those NOT in active_in_window are churned
    labels = (
        eligible_users
        .join(active_in_window, "user_id", "left")
        .withColumn(
            "churned",
            F.when(F.col("churned").isNull(), F.lit(1)).otherwise(F.col("churned"))
        )
    )

    return labels


labels = generate_churn_labels(FEATURE_CUTOFF, MAX_DATE)

# Class balance check
total_labels = labels.count()
churned_count = labels.filter(F.col("churned") == 1).count()
not_churned_count = labels.filter(F.col("churned") == 0).count()

print("=" * 60)
print("  LABEL GENERATION")
print("=" * 60)
print(f"  Eligible customers:   {total_labels:,}")
print(f"  Churned (1):          {churned_count:,} ({churned_count / total_labels:.1%})")
print(f"  Not churned (0):      {not_churned_count:,} ({not_churned_count / total_labels:.1%})")
print(f"  Churn rate:           {churned_count / total_labels:.1%}")
print("=" * 60)


  LABEL GENERATION
  Eligible customers:   62,000
  Churned (1):          57,572 (92.9%)
  Not churned (0):      4,428 (7.1%)
  Churn rate:           92.9%


#### Dataset Assembly

Join all feature groups with labels. Add pipeline metadata columns for lineage tracking:
- `feature_cutoff_date` — which temporal snapshot generated these features
- `prediction_window_days` — the churn definition used
- `feature_version` — schema version for tracking across retrains
- `pipeline_run_timestamp` — when this dataset was created

In [0]:
pipeline_run_ts = datetime.utcnow()

churn_features = (
    labels
    .join(txn_features, "user_id", "inner")
    .join(beh_features, "user_id", "left")
    .join(prod_features, "user_id", "left")
    .join(demo_features, "user_id", "inner")
    # Fill nulls for behavioral features (customers with no pre-cutoff events)
    .fillna({
        "total_sessions": 0,
        "session_count_30d": 0,
        "days_since_last_session": PREDICTION_WINDOW_DAYS * 4,
        "avg_session_depth": 0.0,
        "browse_to_buy_ratio": 0.0,
        "cart_abandonment_rate": 0.0,
    })
    # Pipeline metadata for lineage
    .withColumn("feature_cutoff_date", F.lit(FEATURE_CUTOFF))
    .withColumn("prediction_window_days", F.lit(PREDICTION_WINDOW_DAYS))
    .withColumn("feature_version", F.lit(FEATURE_VERSION))
    .withColumn("pipeline_run_timestamp", F.lit(pipeline_run_ts))
)

print(f"  Feature dataset: {churn_features.count():,} rows, {len(churn_features.columns)} columns")


/home/spark-30df2010-4432-4c95-b4ef-c3/.ipykernel/325/command-6965376355937506-2652569225:1: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  pipeline_run_ts = datetime.utcnow()


  Feature dataset: 62,000 rows, 28 columns


#### Final Validation & Distribution Summary

In [0]:
print("=" * 60)
print("  FINAL DATASET VALIDATION")
print("=" * 60)

all_pass = True
ds = churn_features

# ── 1. No nulls in target ──
null_target = ds.filter(F.col("churned").isNull()).count()
check = null_target == 0
status = "✓" if check else "✗"
if not check:
    all_pass = False
print(f"  {status} No nulls in target ({null_target})")

# ── 2. No nulls in user_id ──
null_uid = ds.filter(F.col("user_id").isNull()).count()
check = null_uid == 0
status = "✓" if check else "✗"
if not check:
    all_pass = False
print(f"  {status} No nulls in user_id ({null_uid})")

# ── 3. Row count matches eligible population ──
ds_count = ds.count()
check = ds_count == eligible_count
status = "✓" if check else "✗"
if not check:
    all_pass = False
print(f"  {status} Row count ({ds_count:,}) == eligible population ({eligible_count:,})")

# ── 4. No duplicate user_ids ──
distinct_users = ds.select("user_id").distinct().count()
check = distinct_users == ds_count
status = "✓" if check else "✗"
if not check:
    all_pass = False
print(f"  {status} No duplicate user_ids ({distinct_users:,} distinct, {ds_count:,} total)")

# ── 5. Class balance sanity ──
churn_rate = ds.filter(F.col("churned") == 1).count() / ds_count
check = 0.05 < churn_rate < 0.95
status = "✓" if check else "✗"
if not check:
    all_pass = False
print(f"  {status} Churn rate ({churn_rate:.1%}) is not degenerate (between 5% and 95%)")

# ── 6. Metadata columns present ──
required_meta = ["feature_cutoff_date", "prediction_window_days", "feature_version", "pipeline_run_timestamp"]
has_meta = all(c in ds.columns for c in required_meta)
status = "✓" if has_meta else "✗"
if not has_meta:
    all_pass = False
print(f"  {status} All metadata columns present")

print(f"\n{'=' * 60}")
if all_pass:
    print("  All validations passed.")
else:
    print("  FAILURES DETECTED. Investigate before exporting.")
print(f"{'=' * 60}")

# ── Distribution summary for key numeric features ──
print(f"\n{'=' * 60}")
print("  KEY FEATURE DISTRIBUTIONS")
print(f"{'=' * 60}")

dist_cols = [
    "order_count", "lifetime_spend", "avg_order_value",
    "days_since_last_order", "total_sessions", "session_count_30d",
    "price_sensitivity_score", "account_age_days",
]

dist_stats = ds.select(
    *[
        F.struct(
            F.round(F.avg(c), 2).alias("mean"),
            F.round(F.expr(f"percentile_approx({c}, 0.5)"), 2).alias("median"),
            F.round(F.expr(f"percentile_approx({c}, 0.95)"), 2).alias("p95"),
            F.round(F.min(c), 2).alias("min"),
            F.round(F.max(c), 2).alias("max"),
        ).alias(c)
        for c in dist_cols
    ]
).collect()[0]

print(f"\n  {'Feature':<28} {'Mean':>10} {'Median':>10} {'P95':>10} {'Min':>10} {'Max':>10}")
print(f"  {'-'*78}")
for c in dist_cols:
    s = dist_stats[c]
    print(f"  {c:<28} {s['mean']:>10} {s['median']:>10} {s['p95']:>10} {s['min']:>10} {s['max']:>10}")


  FINAL DATASET VALIDATION
  ✓ No nulls in target (0)
  ✓ No nulls in user_id (0)
  ✓ Row count (62,000) == eligible population (62,000)
  ✓ No duplicate user_ids (62,000 distinct, 62,000 total)
  ✓ Churn rate (92.9%) is not degenerate (between 5% and 95%)
  ✓ All metadata columns present

  All validations passed.

  KEY FEATURE DISTRIBUTIONS

  Feature                            Mean     Median        P95        Min        Max
  ------------------------------------------------------------------------------
  order_count                        1.42          1          3          1          4
  lifetime_spend                    123.2      79.99     368.95        0.0    1722.93
  avg_order_value                   86.53      60.58     239.99        1.5    1149.99
  days_since_last_order            612.04        441       1766          1       2657
  total_sessions                     2.26          2          5          1         12
  session_count_30d                   0.1          0    

#### Investigation: 23 Missing Customers in Transactional Features

Initial validation flagged a row count mismatch: 62,000 eligible customers but only 61,977 transactional feature rows. We investigated the 23 missing customers. The diagnostic below reveals a temporal lag between order placement and order item creation — orders were placed 1–3 days before the feature cutoff (April 13–16), but their associated order items were created 1–4 days after the cutoff (April 17–20). This is a realistic business event timing pattern, not a data bug. The transactional features join was corrected from `inner` to `left` to retain these customers with zero spend metrics, preserving point-in-time correctness.

In [0]:
# ── DIAGNOSTIC: Who are the missing 23 customers? ──
missing_users = (
    eligible_users
    .join(txn_features.select("user_id"), "user_id", "left_anti")
)

missing_count = missing_users.count()
print(f"Missing users: {missing_count}")
print()

# For each missing user, check their orders and order_items
missing_ids = [row["user_id"] for row in missing_users.collect()]

orders = spark.table(ORDERS)
order_items = spark.table(ORDER_ITEMS)

print("=" * 80)
print("  MISSING USER INVESTIGATION")
print("=" * 80)

for uid in missing_ids[:10]:  # Show first 10 in detail
    print(f"\n  user_id: {uid}")
    
    # Their pre-cutoff orders
    user_orders = (
        orders
        .filter(F.col("user_id") == uid)
        .filter(F.col("status") != "Cancelled")
        .filter(F.col("created_at") < F.lit(FEATURE_CUTOFF))
    )
    pre_cutoff_order_count = user_orders.count()
    
    # Their pre-cutoff order items
    user_items = (
        order_items
        .filter(F.col("user_id") == uid)
        .filter(F.col("status") != "Cancelled")
        .filter(F.col("created_at") < F.lit(FEATURE_CUTOFF))
    )
    pre_cutoff_item_count = user_items.count()
    
    # All their order items (no time filter)
    all_items = (
        order_items
        .filter(F.col("user_id") == uid)
        .filter(F.col("status") != "Cancelled")
    )
    all_item_count = all_items.count()
    
    # Show order and item timestamps
    order_dates = [str(r["created_at"]) for r in user_orders.select("created_at").collect()]
    item_dates = [str(r["created_at"]) for r in all_items.select("created_at").orderBy("created_at").collect()]
    
    print(f"    Pre-cutoff orders:      {pre_cutoff_order_count}")
    print(f"    Pre-cutoff order_items: {pre_cutoff_item_count}")
    print(f"    All order_items:        {all_item_count}")
    print(f"    Order dates:            {order_dates}")
    print(f"    Item dates (all):       {item_dates}")
    
    if pre_cutoff_order_count > 0 and pre_cutoff_item_count == 0:
        print(f"    → CONFIRMED: has orders but NO pre-cutoff items")
    else:
        print(f"    → UNEXPECTED PATTERN — investigate further")

# Summary
print(f"\n{'=' * 80}")
print(f"  SUMMARY")
print(f"{'=' * 80}")
print(f"  Total missing: {missing_count}")
print(f"  Feature cutoff: {FEATURE_CUTOFF}")

Missing users: 23

  MISSING USER INVESTIGATION

  user_id: 379
    Pre-cutoff orders:      1
    Pre-cutoff order_items: 0
    All order_items:        2
    Order dates:            ['2026-04-16 21:30:42']
    Item dates (all):       ['2026-04-17 19:30:58', '2026-04-20 19:55:59']
    → CONFIRMED: has orders but NO pre-cutoff items

  user_id: 48842
    Pre-cutoff orders:      1
    Pre-cutoff order_items: 0
    All order_items:        2
    Order dates:            ['2026-04-16 19:41:03']
    Item dates (all):       ['2026-04-19 17:57:48', '2026-04-20 19:50:57']
    → CONFIRMED: has orders but NO pre-cutoff items

  user_id: 10517
    Pre-cutoff orders:      1
    Pre-cutoff order_items: 0
    All order_items:        2
    Order dates:            ['2026-04-15 06:45:14']
    Item dates (all):       ['2026-04-18 03:36:36', '2026-04-19 06:48:56']
    → CONFIRMED: has orders but NO pre-cutoff items

  user_id: 76812
    Pre-cutoff orders:      1
    Pre-cutoff order_items: 0
    All order_i

#### Export

Two outputs:
1. **Delta table** in Unity Catalog (`ecommerce.gold.churn_feature_dataset`) — queryable from any Databricks notebook.
2. **Parquet file** in the Volume — download to your MacBook for Feast feature store ingestion.

In [0]:
t0 = time.time()

spark.sql("CREATE VOLUME IF NOT EXISTS ecommerce.gold.raw_data")
print("Volume ecommerce.gold.raw_data created.")

# ── 1. Write Delta table ──
churn_features.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(OUTPUT_TABLE)

delta_duration = round(time.time() - t0, 1)
delta_count = spark.table(OUTPUT_TABLE).count()

# ── 2. Write Parquet to Volume for local Feast ingestion ──
PARQUET_PATH = f"/Volumes/{CATALOG}/{GOLD_SCHEMA}/raw_data/churn_feature_dataset.parquet"

t1 = time.time()
churn_features.write.mode("overwrite").parquet(PARQUET_PATH)
parquet_duration = round(time.time() - t1, 1)

print("=" * 60)
print("  EXPORT COMPLETE")
print("=" * 60)
print(f"  Delta table:  {OUTPUT_TABLE}")
print(f"    Rows: {delta_count:,} | Time: {delta_duration}s")
print(f"  Parquet file: {PARQUET_PATH}")
print(f"    Time: {parquet_duration}s")
print(f"\n  Download the Parquet file from Databricks UI:")
print(f"    Catalog → ecommerce → gold → raw_data → churn_feature_dataset.parquet")
print("=" * 60)


Volume ecommerce.gold.raw_data created.
  EXPORT COMPLETE
  Delta table:  ecommerce.gold.churn_feature_dataset
    Rows: 62,000 | Time: 6.8s
  Parquet file: /Volumes/ecommerce/gold/raw_data/churn_feature_dataset.parquet
    Time: 5.5s

  Download the Parquet file from Databricks UI:
    Catalog → ecommerce → gold → raw_data → churn_feature_dataset.parquet


#### Sample: Feature Dataset

In [0]:
display(
    spark.table(OUTPUT_TABLE)
    .select(
        "user_id", "churned",
        "order_count", "lifetime_spend", "avg_order_value",
        "days_since_last_order", "return_rate",
        "total_sessions", "session_count_30d", "cart_abandonment_rate",
        "favorite_category", "price_sensitivity_score",
        "age", "gender", "traffic_source",
        "feature_cutoff_date", "prediction_window_days", "feature_version",
    )
    .limit(10)
)


user_id,churned,order_count,lifetime_spend,avg_order_value,days_since_last_order,return_rate,total_sessions,session_count_30d,cart_abandonment_rate,favorite_category,price_sensitivity_score,age,gender,traffic_source,feature_cutoff_date,prediction_window_days,feature_version
2719,1,1,59.93,59.93,77,0.0,1,0,0.0,Blazers & Jackets,0.0,63,F,Email,2026-04-17,90,v1
3676,1,1,32.0,32.0,2280,0.0,1,0,0.0,Socks & Hosiery,0.0,43,F,Search,2026-04-17,90,v1
5904,1,1,62.0,62.0,544,0.0,1,0,0.0,Active,0.0,48,F,Search,2026-04-17,90,v1
8546,1,2,85.49,42.75,917,0.0,2,0,0.0,Swim,0.0,62,F,Search,2026-04-17,90,v1
11200,1,1,62.99,62.99,383,0.0,2,0,0.5,Sleep & Lounge,0.0,60,F,Search,2026-04-17,90,v1
14355,1,1,172.82,172.82,314,0.0,5,0,0.6154,Sweaters,0.0,66,F,Email,2026-04-17,90,v1
17781,1,4,216.32,54.08,1040,0.0,5,0,0.2857,Sweaters,0.0,70,F,Search,2026-04-17,90,v1
17914,1,1,50.98,50.98,704,0.0,2,0,0.5,Accessories,0.0,61,F,Search,2026-04-17,90,v1
20224,1,1,15.0,15.0,12,0.0,1,1,0.0,Pants & Capris,0.0,51,F,Search,2026-04-17,90,v1
21438,0,1,98.99,98.99,103,0.0,1,0,0.0,Sweaters,0.0,65,F,Search,2026-04-17,90,v1


#### Feature Engineering Summary

| Metric | Value |
|---|---|
| **Eligible customers** | *printed above* |
| **Engineered features** | 22 predictive + 4 metadata |
| **Transactional features** | 7 (order_count, lifetime_spend, avg_order_value, days_since_last_order, first_to_second_order_days, total_items_purchased, return_rate) |
| **Behavioral features** | 6 (total_sessions, session_count_30d, avg_session_depth, browse_to_buy_ratio, cart_abandonment_rate, days_since_last_session) |
| **Product interaction features** | 4 (distinct_products_purchased, distinct_categories_purchased, favorite_category, price_sensitivity_score) |
| **Demographic features** | 5 (age, gender, country, traffic_source, account_age_days) |
| **Target** | `churned` (binary: 1 = no order in prediction window, 0 = active) |
| **Prediction window** | Configurable via `PREDICTION_WINDOW_DAYS` (currently 90 days) |
| **Feature version** | v1 |
| **Output — Delta table** | `ecommerce.gold.churn_feature_dataset` |
| **Output — Parquet** | `/Volumes/ecommerce/gold/raw_data/churn_feature_dataset.parquet` |
| **Leakage prevention** | Temporal cutoff — all features use data strictly before cutoff, labels use data after cutoff |
| **Categorical encoding** | Deferred to training notebook — categoricals preserved as strings |
| **Next step** | Download Parquet → Feast feature store (local) → MLflow training |